In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
)
from sklearn.pipeline import make_pipeline
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import warnings
import joblib
from pathlib import Path

warnings.filterwarnings("ignore")

# Download NLTK data (runs once, then caches locally)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [3]:
# Adjust path to match your local copy of the Kaggle CSV
DATA_PATH = Path("data/raw/fake_job_postings.csv")  # or wherever you saved it

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape}")
print(f"\nFraud distribution:\n{df['fraudulent'].value_counts()}")
print(f"Fraud rate: {df['fraudulent'].mean()*100:.2f}%")


Dataset loaded: (17880, 18)

Fraud distribution:
fraudulent
0    17014
1      866
Name: count, dtype: int64
Fraud rate: 4.84%


In [4]:
def combine_text_fields(row, fields=(
    'title', 'location', 'company_profile', 'description',
    'requirements', 'benefits', 'required_experience',
    'required_education', 'industry', 'function',
)):
    """Combine all text columns into one string."""
    text_parts = []
    for field in fields:
        if pd.notna(row[field]):
            text_parts.append(str(row[field]))
    return " ".join(text_parts) if text_parts else "unknown job"

print("Combining text fields...")
df['combined_text'] = df.apply(combine_text_fields, axis=1)
print("Done")


Combining text fields...
Done


In [5]:
def preprocess_text(text):
    """Tokenize, remove stopwords, lemmatize."""
    if pd.isna(text):
        return ""
    
    # Tokenize
    tokens = word_tokenize(str(text).lower())
    
    # Remove stopwords and non-alphabetic tokens
    stop_words = set(stopwords.words("english"))
    tokens = [token for token in tokens if token.isalpha() and token not in stop_words]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return " ".join(tokens)

print("Preprocessing text...")
df['text_processed'] = df['combined_text'].apply(preprocess_text)
print("Done")


Preprocessing text...
Done


In [6]:
# Location-based fraud ratio (simplified; use your original logic if available)
location_fraud_ratio = df.groupby('location')['fraudulent'].sum() / df.groupby('location').size()
df['location_fraud_ratio'] = df['location'].map(location_fraud_ratio).fillna(0.05)

# Character count
df['character_count'] = df['combined_text'].str.len()


In [7]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    stop_words='english',
)

X_text = vectorizer.fit_transform(df['text_processed'])

print(f"TF-IDF vectorization complete:")
print(f"  Shape: {X_text.shape}")
print(f"  Sparsity: {(1 - X_text.nnz / (X_text.shape[0] * X_text.shape[1]))*100:.1f}%")


TF-IDF vectorization complete:
  Shape: (17880, 5000)
  Sparsity: 96.9%


In [8]:
# Numeric features (adjust to match your original feature list if needed)
numeric_features = [
    'telecommuting',
    'has_company_logo',
    'has_questions',
    'location_fraud_ratio',
    'character_count',
]

X_numeric = df[numeric_features].fillna(0).values
X_numeric_sparse = csr_matrix(X_numeric)

# Combine text + numeric
X_combined = hstack([X_text, X_numeric_sparse])

y = df['fraudulent'].values

print(f"Feature combination complete:")
print(f"  Text features: {X_text.shape[1]}")
print(f"  Numeric features: {len(numeric_features)}")
print(f"  Total features: {X_combined.shape[1]}")
print(f"  Samples: {X_combined.shape[0]}")


Feature combination complete:
  Text features: 5000
  Numeric features: 5
  Total features: 5005
  Samples: 17880


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print(f"Train/Test Split (Stratified):")
print(f"Training set:")
print(f"  - Size: {X_train.shape}")
print(f"  - Fraud rate: {y_train.mean()*100:.2f}%")
print(f"  - Legitimate: {(y_train==0).sum()}")
print(f"  - Fraudulent: {(y_train==1).sum()}")
print(f"Test set:")
print(f"  - Size: {X_test.shape}")
print(f"  - Fraud rate: {y_test.mean()*100:.2f}%")
print(f"  - Legitimate: {(y_test==0).sum()}")
print(f"  - Fraudulent: {(y_test==1).sum()}")


Train/Test Split (Stratified):
Training set:
  - Size: (12516, 5005)
  - Fraud rate: 4.84%
  - Legitimate: 11910
  - Fraudulent: 606
Test set:
  - Size: (5364, 5005)
  - Fraud rate: 4.85%
  - Legitimate: 5104
  - Fraudulent: 260


In [10]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)
y_pred_proba_nb = nb_model.predict_proba(X_test)[:, 1]

cm_nb = confusion_matrix(y_test, y_pred_nb)

print("="*80)
print("NAIVE BAYES - Baseline Model")
print("="*80)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_nb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_nb):.4f}")
print(f"\nConfusion Matrix:\n{cm_nb}")


NAIVE BAYES - Baseline Model
Accuracy:  0.9702
F1-Score:  0.5960
Precision: 0.8676
Recall:    0.4538
ROC-AUC:   0.8494

Confusion Matrix:
[[5086   18]
 [ 142  118]]


In [11]:
from pathlib import Path
import joblib

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(nb_model, MODELS_DIR / "naive_bayes_model.pkl")
joblib.dump(vectorizer, MODELS_DIR / "vectorizer.pkl")

# Also create the pipeline version
X_text_only = df['text_processed']
nb_pipeline = make_pipeline(
    TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.8,
        stop_words='english',
    ),
    MultinomialNB(),
)
nb_pipeline.fit(X_text_only, y)
joblib.dump(nb_pipeline, MODELS_DIR / "nb_pipeline.pkl")

print("Saved:")
print("  - naive_bayes_model.pkl")
print("  - vectorizer.pkl")
print("  - nb_pipeline.pkl")


Saved:
  - naive_bayes_model.pkl
  - vectorizer.pkl
  - nb_pipeline.pkl
